# 22 — Repeatability: the same question several times (jev-lite, Laya, Qwen3.5-4B local vs JEV API)

If a judge is asked exactly the same question again, does it give the same **answer** and the same
**probability of that answer**? Every call records:

- `answer`: the predicted answer (Bool: `yes` / `no` at 0.5; Choice: the option text; Score: the level index),
- `p_answer`: the model's probability **of that answer** (Bool: P(yes) or 1 − P(yes); Choice / Score: the top probability),
- `probs`: the full distribution (Bool: `[P(yes)]`).

Conditions:

- **Local models (MPS), `single`:** the identical request, one record per call, `REPEATS_LOCAL` times.
- **Local models, `batched`:** the same record inside a batch of `BATCH_SIZE` *different* neighbours, reshuffled
  every repeat, as in batched serving (padding and batch composition change; the record does not).
- **JEV (API):** the identical request `JEV_REPEATS` times, in rounds (all records once, then again), 8 requests
  in flight, so repeats are spread over minutes. **Real API calls:** `N_RECORDS × JEV_REPEATS`
  (default 100 × 5 = 500 calls, ~0.25M input tokens). Off by default: set `RUN_JEV = True`. Calls are
  appended to `data/repeatability/jev_calls.jsonl` as they come; a rerun continues where it stopped.

- **Qwen3.5-4B (local LLM, llama.cpp):** through `llama-server` (start it in another terminal with
  `llm -s -Q4 -c 16384`; the notebook skips Qwen if the server is not up). `QWEN_N_RECORDS` of the same questions, `QWEN_REPEATS` times, in each of `QWEN_MODES`:
  - `label t=0`: the answer is one label token (`yes`/`no`, `A`–`Z`, `0`–`6`), forced by a grammar; the
    probabilities come from that token's logprobs, renormalised over the labels — a real distribution, like the classifiers;
  - `json t=0.7` (`json t=0` can be added to `QWEN_MODES`): structured output `{"answer": <label>, "confidence": 0–1}` (JSON schema);
    P(answer) is the model's **stated** confidence, the usual LLM-as-a-judge setup, greedy and sampled.
  Free, but slow (~1 s per call on an M1 Pro): default 50 records × 3 repeats × 2 modes = 300 calls.
  Calls are appended to `data/repeatability/qwen_calls.jsonl`; a rerun continues where it stopped.

Every model's answers are also scored against the labels (`accuracy`), since they come for free.

Results: `data/repeatability/`.

## 1. Environment

In [ ]:
STACK = "apple_silicon"

import os
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
os.environ.setdefault("HF_HUB_OFFLINE", "1")  # all three local models are in the Hugging Face cache

from jevlite.config import apply_env, load_config
from jevlite.log import logger, setup_logging

setup_logging()
cfg = load_config(STACK)
apply_env(cfg)
logger.info("stack {}: config loaded, root {}", STACK, ROOT)

In [ ]:
import platform
import subprocess

import torch


def _sysctl(key: str) -> str:
    return subprocess.run(["sysctl", "-n", key], capture_output=True, text=True).stdout.strip()


ENV = {
    "chip": _sysctl("machdep.cpu.brand_string"),
    "memory_gb": round(int(_sysctl("hw.memsize") or 0) / 2**30),
    "macos": platform.mac_ver()[0],
    "python": platform.python_version(),
    "torch": torch.__version__,
    "mps": torch.backends.mps.is_available(),
}
assert ENV["mps"], "MPS is not available: this notebook is for Apple Silicon"
ENV

## 2. Parameters

In [ ]:
SAMPLE = "data/jev_api/sample.jsonl"
JEVLITE_MODEL = "data/runs/stage_a-full-s42/final"

LOCAL_MODELS = ["jevlite", "laya"]  # NLI left out: deterministic like these two and ~10x slower (one pass per option)
N_RECORDS = 100           # questions per model, split evenly over Bool / Choice / Score (34 / 33 / 33)
SEED = 0
REPEATS_LOCAL = 10
MODES = ["single", "batched"]
BATCH_SIZE = 16           # batched mode: records per forward pass

RUN_JEV = False           # True sends N_RECORDS * JEV_REPEATS API calls
JEV_REPEATS = 5
JEV_CONCURRENCY = 8
JEV_MODEL = "jev-latest"

RUN_QWEN = True           # needs llama-server: `llm -s -Q4 -c 16384` in another terminal; skipped if not up
QWEN_URL = "http://127.0.0.1:8080"
QWEN_N_RECORDS = 50       # the first of the N_RECORDS questions, split evenly over the primitives (17 / 17 / 16)
QWEN_REPEATS = 3
QWEN_MODES = [("label", 0.0), ("json", 0.7)]  # (mode, temperature); ("json", 0.0) is greedy and repeats itself
# 50 records x 3 repeats x 2 modes = 300 calls
QWEN_TOP_LOGPROBS = 50    # label mode: alternatives returned per token (covers up to 28 options)
QWEN_TIMEOUT = 300        # seconds per request

OUT = Path("data/repeatability")
OUT.mkdir(parents=True, exist_ok=True)

## 3. Records

In [ ]:
import random

import numpy as np
import pandas as pd

from jevlite.data.unified import load_records

records = list(load_records(SAMPLE))
rng = random.Random(SEED)
bench = []
for i, prim in enumerate(("bool", "choice", "score")):
    pool = sorted((r for r in records if r.type == prim), key=lambda r: r.id)
    n = N_RECORDS // 3 + (i < N_RECORDS % 3)
    bench += rng.sample(pool, min(n, len(pool)))
by_id = {r.id: r for r in bench}
pd.Series([r.type for r in bench]).value_counts()

## 4. Answer and probability of the answer

`call_row` turns one model output into the stored row. The batched functions are those of notebook 21
(checked there to match `predict` to 1e-6).

In [ ]:
from jevlite.baselines import load_baseline
from jevlite.inference import _normalise


def call_row(model: str, mode: str, repeat: int, r, p) -> dict:
    p = np.asarray(p, dtype=float)
    if r.type == "bool":
        yes = float(p[0])
        answer, p_answer, margin = ("yes" if yes >= 0.5 else "no"), max(yes, 1 - yes), abs(2 * yes - 1)
    else:
        top = np.sort(p)[::-1]
        k = int(np.argmax(p))
        answer = r.candidates[k] if r.type == "choice" else str(k)
        p_answer, margin = float(top[0]), float(top[0] - (top[1] if len(top) > 1 else 0.0))
    return {"model": model, "mode": mode, "repeat": repeat, "id": r.id, "primitive": r.type,
            "answer": answer, "p_answer": p_answer, "margin": margin, "probs": p.tolist()}


def target_answer(r) -> str:
    if r.type == "bool":
        return "yes" if float(r.target) >= 0.5 else "no"
    k = int(np.argmax(r.target)) if isinstance(r.target, list) else int(r.target)
    return r.candidates[k] if r.type == "choice" else str(k)


def jevlite_batch(model, recs):
    rows, _ = model.predictor._probs([(r.state, r.to_question(f"q{j}")) for j, r in enumerate(recs)])
    return [np.asarray(p if r.type == "bool" else _normalise(p)) for r, p in zip(recs, rows)]


def laya_batch(model, recs):
    from jevlite.baselines.laya import to_laya_question

    rl, items, qs = model.rl, [], []
    for r in recs:
        q = to_laya_question(r.to_question("q"))
        ids, markers = rl.build_sequence(model.tokenizer, r.state, q, model.cfg["max_len"], model.cfg["head_max_len"])
        if len(markers) != len(rl.render_options(q)):
            raise ValueError("options do not fit in Laya's header")
        items.append({"ids": ids, "markers": markers, "qtype": rl.QTYPES[q["t"]], "target": [0.0] * len(markers),
                      "label": -1, "episode": 0, "ep_step": 0, "ep_len": 1, "src": "jevlite"})
        qs.append(q)
    b = rl.collate_items([items], model.tokenizer.pad_token_id)
    with torch.no_grad():
        logits, _ = model.model(b["input_ids"].to(model.device), b["attention_mask"].to(model.device),
                                b["marker_pos"].to(model.device), b["marker_mask"].to(model.device), b["qtype"].to(model.device))
    logits = logits.float().cpu().numpy()
    out = []
    for i, (item, q) in enumerate(zip(items, qs)):
        k = len(item["markers"])
        z = logits[i, :k] / model._temperature(item["qtype"], k)
        p = np.exp(z - z.max())
        p /= p.sum()
        out.append(np.array([p[1]]) if q["t"] == "noul" else p)
    return out


BATCH_FN = {"jevlite": jevlite_batch, "laya": laya_batch}

## 5. Local models

In [ ]:
import gc
import time


def free_mps():
    gc.collect()
    torch.mps.empty_cache()


def run_local(name: str) -> list[dict]:
    free_mps()
    model = load_baseline(name, JEVLITE_MODEL if name == "jevlite" else None, "mps")
    fn, rows = BATCH_FN[name], []
    for r in bench[:5]:  # warm-up
        try:
            model.predict(r.state, [r.to_question("q")])
        except ValueError:
            pass
    for rep in range(REPEATS_LOCAL):
        if "single" in MODES:
            for r in bench:
                try:
                    rows.append(call_row(name, "single", rep, r, model.predict(r.state, [r.to_question("q")])[0]))
                except ValueError:  # Laya: options do not fit
                    pass
        if "batched" in MODES:
            order = bench[:]
            random.Random(f"{SEED}:{rep}").shuffle(order)  # new neighbours every repeat
            for s in range(0, len(order), BATCH_SIZE):
                chunk = order[s : s + BATCH_SIZE]
                try:
                    outs = list(zip(chunk, fn(model, chunk)))
                except ValueError:
                    outs = []
                    for r in chunk:
                        try:
                            outs.append((r, fn(model, [r])[0]))
                        except ValueError:
                            pass
                rows += [call_row(name, "batched", rep, r, p) for r, p in outs]
    logger.info("{}: {} calls", name, len(rows))
    del model
    free_mps()
    return rows


calls = []
for name in LOCAL_MODELS:
    calls += run_local(name)
len(calls)

## 6. JEV (API)

Rounds of all records; each call is appended to `jev_calls.jsonl` with the model version and request id.
A rerun skips the (record, repeat) pairs already there, so an interrupted run does not pay twice.

In [ ]:
import json

JEV_FILE = OUT / "jev_calls.jsonl"
jev_rows = [json.loads(l) for l in JEV_FILE.open()] if JEV_FILE.exists() else []

if RUN_JEV:
    import asyncio
    import sys

    from dotenv import load_dotenv
    from typesafe_sdk import AsyncTypeSafeClient

    sys.path.insert(0, str(ROOT / "scripts"))
    import predict_jev_api as jev_api

    load_dotenv()
    done = {(c["id"], c["repeat"]) for c in jev_rows}
    sem = asyncio.Semaphore(JEV_CONCURRENCY)
    errors = 0

    async def one(client, r, rep, f):
        global errors
        async with sem:
            try:
                resp = await client.system_one(state=r.state, questions={"q": jev_api._question(r)})
            except Exception as e:
                errors += 1
                logger.warning("{} repeat {}: {}", r.id, rep, str(e)[:120])
                return
        p = jev_api._probs(r.to_question("q"), resp.answers["q"])
        row = call_row("jev", "api", rep, r, p) | {"version": resp.model, "request_id": resp.request_id,
                                                    "input_tokens": resp.usage.input_tokens}
        f.write(json.dumps(row) + "\n")
        f.flush()
        jev_rows.append(row)

    async with AsyncTypeSafeClient(api_key=os.environ["TYPESAFE_API_KEY"], model=JEV_MODEL) as client:
        with JEV_FILE.open("a") as f:
            for rep in range(JEV_REPEATS):
                todo = [r for r in bench if (r.id, rep) not in done]
                await asyncio.gather(*(one(client, r, rep, f) for r in todo))
                logger.info("JEV round {}: {} calls, {} errors so far", rep + 1, len(todo), errors)

if jev_rows:
    calls += [c for c in jev_rows if c["id"] in by_id]
    logger.info("JEV: {} calls, versions {}", len(jev_rows), pd.Series([c.get("version") for c in jev_rows]).value_counts().to_dict())
else:
    logger.info("no JEV calls (RUN_JEV is False and {} does not exist)", JEV_FILE)

## 6b. Qwen3.5-4B (llama.cpp, local)

The prompt gives the text, the question and the labelled options / levels. `label` mode lets the model write
exactly one label token (grammar) and reads the probabilities of the labels from that token's top logprobs
(raw, before sampling); `label_mass` is the share of the probability that fell on the labels at all.
`json` mode asks for `{"answer", "confidence"}` under a JSON schema; the stored distribution puts the stated
confidence on the answer and spreads the rest evenly, so `p_answer` is the stated confidence.

In [ ]:
import json
import math
import string
import urllib.request

from jevlite.encoding import serialize_state

QWEN_FILE = OUT / "qwen_calls.jsonl"
SYSTEM = "You are a careful judge. Answer only from the given text and question."
LETTERS = list(string.ascii_uppercase + string.ascii_lowercase)


def qwen_labels(r) -> list[str]:
    if r.type == "bool":
        return ["yes", "no"]
    if r.type == "choice":
        return LETTERS[: len(r.candidates)]
    return [str(i) for i in range(len(r.candidates))]


def qwen_prompt(r) -> str:
    if r.type == "bool":
        ask = f"Statement: {r.prompt}\n\nIs the statement true according to the text? Answer yes or no."
    elif r.type == "choice":
        opts = "\n".join(f"{l}. {o}" for l, o in zip(qwen_labels(r), r.candidates))
        ask = f"{r.prompt}\n\nOptions:\n{opts}\n\nAnswer with the letter of the best option."
    else:
        levels = "\n".join(f"{i}: {o}" for i, o in enumerate(r.candidates))
        ask = f"{r.prompt}\n\nLevels, from low to high:\n{levels}\n\nAnswer with the number of the level."
    return f"Text:\n{serialize_state(r.state)}\n\n{ask}"


def qwen_post(path: str, body: dict) -> dict:
    req = urllib.request.Request(QWEN_URL + path, data=json.dumps(body).encode(), headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=QWEN_TIMEOUT) as resp:
        return json.loads(resp.read())


def qwen_up() -> bool:
    try:
        with urllib.request.urlopen(QWEN_URL + "/health", timeout=5) as resp:
            return resp.status == 200
    except Exception:
        return False


def as_answer(r, label: str) -> str:
    labels = qwen_labels(r)
    if r.type == "bool":
        return label
    k = labels.index(label)
    return r.candidates[k] if r.type == "choice" else str(k)


def qwen_label_call(r, temperature: float) -> dict:
    labels = qwen_labels(r)
    body = {"messages": [{"role": "system", "content": SYSTEM}, {"role": "user", "content": qwen_prompt(r)}],
            "temperature": temperature, "max_tokens": 1, "grammar": "root ::= " + " | ".join(f'"{l}"' for l in labels),
            "logprobs": True, "top_logprobs": QWEN_TOP_LOGPROBS, "chat_template_kwargs": {"enable_thinking": False}}
    out = qwen_post("/v1/chat/completions", body)
    mass = dict.fromkeys(labels, 0.0)
    for t in out["choices"][0]["logprobs"]["content"][0]["top_logprobs"]:
        tok = t["token"].strip()
        tok = tok.lower() if r.type == "bool" else tok  # yes / Yes are the same answer; A and a are not
        if tok in mass:
            mass[tok] += math.exp(t["logprob"])
    total = sum(mass.values())
    if total == 0:
        raise ValueError("no label among the top logprobs")
    p = np.array([mass[l] for l in labels]) / total
    return {"probs": p[:1] if r.type == "bool" else p, "label_mass": total,
            "generated": out["choices"][0]["message"]["content"].strip()}


def qwen_json_call(r, temperature: float) -> dict:
    labels = qwen_labels(r)
    schema = {"type": "object", "required": ["answer", "confidence"], "properties": {
        "answer": {"type": "string", "enum": labels}, "confidence": {"type": "number", "minimum": 0, "maximum": 1}}}
    user = qwen_prompt(r) + "\n\nReply in JSON with your answer and your confidence (0 to 1) that it is correct."
    body = {"messages": [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}],
            "temperature": temperature, "max_tokens": 64, "chat_template_kwargs": {"enable_thinking": False},
            "response_format": {"type": "json_schema", "json_schema": {"name": "judgement", "schema": schema}}}
    out = qwen_post("/v1/chat/completions", body)
    obj = json.loads(out["choices"][0]["message"]["content"])
    label, conf = obj["answer"], min(max(float(obj["confidence"]), 0.0), 1.0)
    if r.type == "bool":
        probs = np.array([conf if label == "yes" else 1 - conf])
    else:
        probs = np.full(len(labels), (1 - conf) / max(len(labels) - 1, 1))
        probs[labels.index(label)] = conf
    return {"probs": probs, "label": label, "confidence": conf}


qwen_bench = []
for i, prim in enumerate(("bool", "choice", "score")):
    n = QWEN_N_RECORDS // 3 + (i < QWEN_N_RECORDS % 3)
    qwen_bench += [r for r in bench if r.type == prim][:n]
qwen_ids = {r.id for r in qwen_bench}
logger.info("Qwen: {} records x {} repeats x {} modes = {} calls planned", len(qwen_bench), QWEN_REPEATS, len(QWEN_MODES),
            len(qwen_bench) * QWEN_REPEATS * len(QWEN_MODES))

qwen_rows = [json.loads(l) for l in QWEN_FILE.open()] if QWEN_FILE.exists() else []

if RUN_QWEN and not qwen_up():
    logger.warning("llama-server is not up at {}: start it with `llm -s -Q4 -c 16384`; Qwen is skipped", QWEN_URL)
elif RUN_QWEN:
    multi = [l for l in sorted({l for r in qwen_bench for l in qwen_labels(r)}) if len(qwen_post("/tokenize", {"content": l})["tokens"]) != 1]
    if multi:
        logger.warning("labels that are not one token (label mode skips their records): {}", multi)
    done = {(c["mode"], c["id"], c["repeat"]) for c in qwen_rows}
    with QWEN_FILE.open("a") as f:
        for mode, temperature in QWEN_MODES:
            key = f"{mode} t={temperature:g}"
            for rep in range(QWEN_REPEATS):
                n = errors = 0
                for r in qwen_bench:
                    if (key, r.id, rep) in done:
                        continue
                    if mode == "label" and set(qwen_labels(r)) & set(multi):
                        continue
                    t = time.perf_counter()
                    try:
                        res = (qwen_label_call if mode == "label" else qwen_json_call)(r, temperature)
                    except Exception as e:
                        errors += 1
                        logger.warning("qwen {} {} repeat {}: {}", key, r.id, rep, str(e)[:120])
                        continue
                    row = call_row("qwen", key, rep, r, res["probs"]) | {"latency_ms": (time.perf_counter() - t) * 1000}
                    if mode == "json":  # the stated answer and confidence, not the argmax of the spread-out distribution
                        row |= {"answer": as_answer(r, res["label"]), "p_answer": res["confidence"],
                                "margin": abs(2 * res["confidence"] - 1)}
                    else:
                        row |= {"label_mass": res["label_mass"], "generated": res["generated"]}
                    f.write(json.dumps(row) + "\n")
                    f.flush()
                    qwen_rows.append(row)
                    n += 1
                logger.info("qwen {} repeat {}: {} calls, {} errors", key, rep + 1, n, errors)

if qwen_rows:
    calls += [c for c in qwen_rows if c["id"] in qwen_ids and c["mode"] in {f"{m} t={t:g}" for m, t in QWEN_MODES}]
    q = pd.DataFrame(qwen_rows)
    logger.info("Qwen: {} calls, median latency {:.0f} ms; label mode: mean label mass {:.3f}", len(q),
                q["latency_ms"].median(), q["label_mass"].mean() if "label_mass" in q else float("nan"))

## 7. Per record: how much the answer and its probability move

For each model × mode × record over its repeats:

- `n_answers`: distinct answers given (1 = always the same),
- `agree`: share of repeats that give the most common answer,
- `p_modal`: the probability of the most common answer in each repeat — its `range` (max − min) and `std`,
- `p_answer_mean`: mean probability of the answer actually given,
- `accuracy`: share of repeats whose answer matches the label.

In [ ]:
df = pd.DataFrame(calls)
df["key"] = df["model"] + " · " + df["mode"]


def per_record(g: pd.DataFrame) -> pd.Series:
    counts = g["answer"].value_counts()
    modal = counts.index[0]
    r = by_id[g.name[1]]
    if r.type == "bool":
        p_modal = g["probs"].map(lambda p: p[0] if modal == "yes" else 1 - p[0])
    else:
        k = r.candidates.index(modal) if r.type == "choice" else int(modal)
        p_modal = g["probs"].map(lambda p: p[k])
    target = target_answer(r)
    return pd.Series({
        "primitive": r.type, "repeats": len(g), "n_answers": len(counts), "agree": counts.iloc[0] / len(g),
        "accuracy": (g["answer"] == target).mean(), "modal_correct": float(modal == target),
        "modal_answer": modal, "p_modal_mean": p_modal.mean(), "p_range": p_modal.max() - p_modal.min(),
        "p_std": p_modal.std(ddof=0), "p_answer_mean": g["p_answer"].mean(), "margin_mean": g["margin"].mean(),
    })


rec = df.groupby(["key", "id"]).apply(per_record, include_groups=False).reset_index()
rec.head()

In [ ]:
MODEL_ORDER = ["nli", "laya", "jevlite", "jev", "qwen"]  # colour slots as in STAGE_A_API.md (NLI keeps its slot, unused), Qwen next
ORDER = [k for m in MODEL_ORDER for k in sorted(rec["key"].unique()) if k.startswith(m + " ·")]

summary = rec.groupby(["key", "primitive"]).agg(
    records=("id", "size"),
    repeats=("repeats", "median"),
    answer_changed=("n_answers", lambda s: (s > 1).mean()),
    agree_mean=("agree", "mean"),
    accuracy=("accuracy", "mean"),
    p_answer_mean=("p_answer_mean", "mean"),
    p_range_mean=("p_range", "mean"),
    p_range_p90=("p_range", lambda s: s.quantile(0.9)),
    p_range_max=("p_range", "max"),
    p_std_mean=("p_std", "mean"),
).reindex(ORDER, level="key")
summary.style.format({"answer_changed": "{:.1%}", "agree_mean": "{:.3f}", "accuracy": "{:.3f}", "p_answer_mean": "{:.3f}",
                      "p_range_mean": "{:.2e}", "p_range_p90": "{:.2e}", "p_range_max": "{:.2e}", "p_std_mean": "{:.2e}"})

## 8. Near the threshold

A probability that wobbles by ±0.01 changes the answer only when the answer is close to a tie. Share of
records whose answer changed, by the mean margin (Bool: |2·P(yes) − 1|; Choice / Score: top − second probability).

In [ ]:
BINS = [0, 0.05, 0.1, 0.2, 0.4, 1.0001]
rec["margin_bin"] = pd.cut(rec["margin_mean"], BINS, right=False,
                           labels=["< 0.05", "0.05–0.1", "0.1–0.2", "0.2–0.4", "≥ 0.4"])
near = rec.pivot_table(index="key", columns="margin_bin", values="n_answers",
                       aggfunc=lambda s: f"{(s > 1).mean():.0%} of {len(s)}" if len(s) else "–", observed=False).reindex(ORDER)
near

In [ ]:
changed = rec[rec["n_answers"] > 1].sort_values(["key", "agree"])
examples = []
for _, row in changed.iterrows():
    g = df[(df["key"] == row["key"]) & (df["id"] == row["id"])].sort_values("repeat")
    examples.append({"model": row["key"], "id": row["id"], "primitive": row["primitive"],
                     "answers (P of answer)": "  ".join(f"{a[:20]} ({p:.2f})" for a, p in zip(g["answer"], g["p_answer"]))})
pd.DataFrame(examples).head(30) if examples else "no record changed its answer in any model"

## 9. Save

In [ ]:
df.drop(columns=["key"]).to_json(OUT / "calls.jsonl", orient="records", lines=True)
rec.to_csv(OUT / "per_record.csv", index=False)
(OUT / "summary.json").write_text(json.dumps({
    "env": ENV, "n_records": len(bench), "repeats_local": REPEATS_LOCAL, "modes": MODES, "batch_size": BATCH_SIZE,
    "jev": {"calls": len(jev_rows), "repeats": JEV_REPEATS},
    "qwen": {"calls": len(qwen_rows), "repeats": QWEN_REPEATS, "modes": QWEN_MODES},
    "summary": summary.reset_index().to_dict("records"),
}, indent=1, default=str))
print(f"→ {OUT}/calls.jsonl, per_record.csv, summary.json")

## 10. Charts

1. Spread of the probability of the (most common) answer across repeats: median and 90th percentile of the
   range per record, log scale; values below 1e-9 are drawn at 1e-9 (identical to float precision).
2. Share of records whose answer changed at least once.

In [ ]:
from matplotlib.lines import Line2D
from matplotlib.ticker import FixedLocator, NullLocator

from jevlite.reports.style import MUTED, SURFACE, TEXT, TEXT_SECONDARY, save_figure, series_colors, setup_matplotlib

plt = setup_matplotlib()
COLORS = series_colors(MODEL_ORDER)
FLOOR = 1e-9
rows = [(k, p) for k in ORDER for p in ("bool", "choice", "score") if ((rec["key"] == k) & (rec["primitive"] == p)).any()]

fig, ax = plt.subplots(figsize=(7.6, 0.3 * len(rows) + 1.8))
for i, (k, p) in enumerate(rows):
    s = rec[(rec["key"] == k) & (rec["primitive"] == p)]["p_range"].clip(lower=FLOOR)
    color = COLORS[k.split(" · ")[0]]
    ax.plot([s.median(), s.quantile(0.9)], [i, i], color=color, linewidth=2, zorder=1)
    ax.scatter([s.median()], [i], s=64, color=color, edgecolor=SURFACE, linewidth=2, zorder=2)
    ax.scatter([s.quantile(0.9)], [i], s=40, color=SURFACE, edgecolor=color, linewidth=2, zorder=2)
ax.set_xscale("log")
ticks = [1e-9, 1e-7, 1e-5, 1e-3, 1e-2, 1e-1, 1]
ax.xaxis.set_major_locator(FixedLocator(ticks))
ax.xaxis.set_minor_locator(NullLocator())
ax.set_xticklabels(["≤1e-9", "1e-7", "1e-5", "0.001", "0.01", "0.1", "1"])
ax.set_xlim(FLOOR / 3, 1.5)
ax.set_yticks(range(len(rows)), [f"{k} · {p}" for k, p in rows])
ax.set_ylim(len(rows) - 0.3, -0.7)
ax.grid(axis="y", visible=False)
ax.tick_params(axis="y", length=0)
ax.set_xlabel("range of P(answer) across repeats, per record (log scale)")
ax.set_title("How much the probability of the answer moves when the question is repeated", loc="left")
handles = [Line2D([], [], marker="o", linestyle="", markersize=8, color=MUTED, label="median record"),
           Line2D([], [], marker="o", linestyle="", markersize=7, markerfacecolor=SURFACE, markeredgecolor=MUTED,
                  markeredgewidth=2, label="90th percentile")]
ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, -0.9 / (0.3 * len(rows) + 1.8)), ncol=2)
print(save_figure(fig, OUT, "p_answer_spread")[0])
fig

In [ ]:
fig, ax = plt.subplots(figsize=(7.6, 0.3 * len(rows) + 1.6))
for i, (k, p) in enumerate(rows):
    s = rec[(rec["key"] == k) & (rec["primitive"] == p)]["n_answers"]
    v = (s > 1).mean()
    ax.barh(i, v, height=0.6, color=COLORS[k.split(" · ")[0]], edgecolor=SURFACE, linewidth=2)
    ax.text(v, i, f"  {v:.0%}  ({(s > 1).sum()} of {len(s)})", va="center", fontsize=9, color=TEXT_SECONDARY)
ax.set_yticks(range(len(rows)), [f"{k} · {p}" for k, p in rows])
ax.set_ylim(len(rows) - 0.5, -0.5)
ax.set_xlim(0, min(1.0, max(0.1, rec["n_answers"].gt(1).groupby([rec["key"], rec["primitive"]]).mean().max() * 1.35)))
ax.grid(axis="y", visible=False)
ax.tick_params(axis="y", length=0)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
ax.set_xlabel("records whose answer changed in at least one repeat")
ax.set_title("Same question, different answer", loc="left")
print(save_figure(fig, OUT, "answer_changed")[0])
fig

## 11. Reading the numbers

- **Local models** are classifiers without sampling or dropout at inference: `single` should be identical
  (range 0). `batched` can move by float rounding (~1e-7 – 1e-6) because padding and neighbours change
  the GPU kernels; that should never flip an answer unless the margin is ~0.
- **JEV** is an API: the server may batch requests, run on different hardware, or change between calls.
  Its spread is what a client sees; compare it with the margin table — a wobble of ±0.01 matters only
  for records within ~0.02 of a tie.
- **Qwen `label t=0`** reads the label distribution from logprobs, so it is comparable with the classifiers;
  llama.cpp with prompt caching and Metal kernels can still move it slightly between calls.
- **Qwen `json`** shows the usual LLM-judge setup: at t=0.7 the answer itself is sampled, so it changes on
  close calls, and the stated confidence is coarse (0.8, 0.9, 0.95 …) rather than a calibrated probability.
- `p_answer_mean` is the confidence the model reports for its own answer; `agree` is how often it repeats it;
  `accuracy` is against the labels of the test set.
- `answer_changed` counts a record once if *any* repeat disagrees, so it grows with the number of repeats.